In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("run_id", "manual")
run_id = dbutils.widgets.get("run_id")

In [0]:
df_pii = (
    spark.read
    .table("databricksbootcamp.greeengridcasestudy.bronze_pii")
    .drop("ingested_at", "run_id")
)

df_plans = (
    spark.read
    .table("databricksbootcamp.greeengridcasestudy.bronze_plans")
    .drop("ingested_at", "run_id")
)

df_readings = (
    spark.read
    .table("databricksbootcamp.greeengridcasestudy.bronze_readings")
    .filter(
        (col("kwh") >= 0) &
        (col("kwh") <= 100)
    )
    .drop("ingested_at", "run_id")
)

In [0]:
df_join = (
    df_readings.join(df_pii, df_readings.meter_id == df_pii.MeterID, "left")
    .drop("MeterID")
    .join(df_plans, df_pii.PlanID == df_plans.PlanID, "left")
    .drop(df_pii.PlanID)
)

In [0]:
df_final = (
    df_join.withColumn("Hour", hour(col("timestamp")))
            .withColumn("FinalRate", 
                when((col("Hour") >= 18) & (col("Hour") <= 21), col("BasePricePerKwh") * 2)
                .otherwise(col("BasePricePerKwh"))
            )
            .withColumn("Bill_Amount", col("FinalRate") * col("kwh"))
            .withColumn("CustomerHash", sha2(col("CustomerEmail"), 256)).drop("CustomerEmail")
            .withColumn("ingested_at", current_timestamp())
            .withColumn("run_id", lit(run_id))
)

(
df_final.write.mode("overwrite")
    .saveAsTable("databricksbootcamp.greeengridcasestudy.silver_smartbill")
)